# Procesamiento de datos - Alura Agente Moyano Demo

Este notebook carga los CSV sintéticos, valida relaciones, crea tablas enriquecidas y genera documentos estructurados para una etapa posterior de embeddings.


## Uso en Colab

Si abrís este notebook en Colab desde GitHub, primero cloná el repositorio y entrá en la carpeta del proyecto. Cambiá la URL por la de tu repo si hace falta.


In [ ]:
# Ejecutar solo si estás en Colab y todavía no clonaste el repo
# !git clone https://github.com/TU_USUARIO/alura-agente-moyano.git
# %cd /content/alura-agente-moyano

!pwd
!ls data/raw


## 1. Carga de datos


In [ ]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path('data/raw')

pacientes = pd.read_csv(RAW_DIR / 'pacientes.csv')
internaciones = pd.read_csv(RAW_DIR / 'internaciones.csv')
movimientos = pd.read_csv(RAW_DIR / 'movimientos.csv')

print('Pacientes:', pacientes.shape)
print('Internaciones:', internaciones.shape)
print('Movimientos:', movimientos.shape)


In [ ]:
display(pacientes.head())
display(internaciones.head())
display(movimientos.head())


## 2. Normalización y fechas


In [ ]:
def limpiar_columnas(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.upper()
        .str.replace(' ', '_', regex=False)
    )
    return df

def limpiar_textos(df):
    df = df.copy()
    cols = df.select_dtypes(include=['object', 'string']).columns
    for col in cols:
        df[col] = df[col].astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)
    return df

pacientes = limpiar_textos(limpiar_columnas(pacientes))
internaciones = limpiar_textos(limpiar_columnas(internaciones))
movimientos = limpiar_textos(limpiar_columnas(movimientos))

pacientes['FENAC'] = pd.to_datetime(pacientes['FENAC'], errors='coerce')
internaciones['FECHA_INGRESO'] = pd.to_datetime(internaciones['FECHA_INGRESO'], errors='coerce')
internaciones['FECHA_EGRESO'] = pd.to_datetime(internaciones['FECHA_EGRESO'], errors='coerce')
movimientos['FECHA_MOVIMIENTO'] = pd.to_datetime(movimientos['FECHA_MOVIMIENTO'], errors='coerce')


## 3. Validaciones básicas


In [ ]:
validaciones = {
    'pacientes': len(pacientes),
    'internaciones': len(internaciones),
    'movimientos': len(movimientos),
    'chist_duplicados_pacientes': pacientes['CHIST'].duplicated().sum(),
    'id_internacion_duplicados': internaciones['ID_INTERNACION'].duplicated().sum(),
    'id_movimiento_duplicados': movimientos['ID_MOVIMIENTO'].duplicated().sum(),
    'internaciones_sin_paciente': (~internaciones['CHIST'].isin(pacientes['CHIST'])).sum(),
    'movimientos_sin_internacion': (~movimientos['ID_INTERNACION'].isin(internaciones['ID_INTERNACION'])).sum(),
}

validaciones


## 4. Tabla enriquecida


In [ ]:
internaciones['DIAS_INTERNACION'] = (internaciones['FECHA_EGRESO'] - internaciones['FECHA_INGRESO']).dt.days
internaciones['ANIO_INGRESO'] = internaciones['FECHA_INGRESO'].dt.year
internaciones['MES_INGRESO'] = internaciones['FECHA_INGRESO'].dt.month
movimientos['ANIO_MOVIMIENTO'] = movimientos['FECHA_MOVIMIENTO'].dt.year
movimientos['MES_MOVIMIENTO'] = movimientos['FECHA_MOVIMIENTO'].dt.month

internaciones_completas = internaciones.merge(pacientes, on='CHIST', how='left')
movimientos_enriquecidos = movimientos.merge(internaciones_completas, on='ID_INTERNACION', how='left')

display(movimientos_enriquecidos.head())
print(movimientos_enriquecidos.shape)


## 5. Consultas de prueba


In [ ]:
print('Internaciones activas:', internaciones['FECHA_EGRESO'].isna().sum())
print('Tipos de movimiento:')
display(movimientos['TIPO_MOVIMIENTO'].value_counts())
print('Diagnósticos más frecuentes:')
display(internaciones['DIAGNOSTICO'].value_counts().head(10))
print('Pabellones con más movimientos:')
display(movimientos['PABELLON'].value_counts().head(10))


## 6. Crear documentos estructurados


In [ ]:
def safe_value(value):
    if pd.isna(value):
        return ''
    return str(value).strip()

def fila_a_texto(row):
    return (
        f"Movimiento {safe_value(row.get('ID_MOVIMIENTO'))}. "
        f"Internación {safe_value(row.get('ID_INTERNACION'))}. "
        f"Paciente demo CHIST {safe_value(row.get('CHIST'))}. "
        f"Fecha de movimiento: {safe_value(row.get('FECHA_MOVIMIENTO'))}. "
        f"Tipo de movimiento: {safe_value(row.get('TIPO_MOVIMIENTO'))}. "
        f"Pabellón: {safe_value(row.get('PABELLON'))}. "
        f"Diagnóstico: {safe_value(row.get('DIAGNOSTICO'))}. "
        f"Estado: {safe_value(row.get('ESTADO'))}."
    )

documentos = []
for idx, row in movimientos_enriquecidos.iterrows():
    documentos.append({
        'page_content': fila_a_texto(row),
        'metadata': {
            'source': 'movimientos_enriquecidos.csv',
            'row': int(idx) + 2,
            'id_movimiento': safe_value(row.get('ID_MOVIMIENTO')),
            'id_internacion': safe_value(row.get('ID_INTERNACION')),
            'chist': safe_value(row.get('CHIST')),
        }
    })

documentos[0]


## 7. Guardar procesados


In [ ]:
import json
PROCESSED_DIR = Path('data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pacientes.to_csv(PROCESSED_DIR / 'pacientes_procesados.csv', index=False)
internaciones.to_csv(PROCESSED_DIR / 'internaciones_procesadas.csv', index=False)
movimientos.to_csv(PROCESSED_DIR / 'movimientos_procesados.csv', index=False)
movimientos_enriquecidos.to_csv(PROCESSED_DIR / 'movimientos_enriquecidos.csv', index=False)

with open(PROCESSED_DIR / 'documentos_movimientos.jsonl', 'w', encoding='utf-8') as f:
    for doc in documentos:
        f.write(json.dumps(doc, ensure_ascii=False) + '\n')

!ls data/processed
